# 🟠 Kafka — Parte 4: Confiabilidade e Tempo em Streaming

---

> **Pré-requisito:** Kafka rodando (Parte 1). kafka-python instalado (Parte 2).

> **Objetivo:** Compreender as garantias de entrega, semânticas de processamento e os modelos de tempo em sistemas de streaming.

| Etapa | O que faremos |
|---|---|
| **1. Semânticas de entrega** | At-most-once, at-least-once, exactly-once |
| **2. Event Time vs Processing Time** | Demonstrar a diferença e o impacto |
| **3. Eventos tardios (Late Events)** | Watermarks e estratégias de tratamento |
| **4. Dead Letter Queue** | Isolando eventos inválidos |
| **5. Idempotência** | Producer idempotente e deduplicação |

In [ ]:
# ============================================================
# ETAPA 1 — SETUP KAFKA (execute sempre que abrir o notebook)
# ============================================================
import subprocess, os, time, logging, sys

logging.getLogger("kafka").setLevel(logging.CRITICAL)

KAFKA_VERSION = "3.7.0"
SCALA_VERSION = "2.13"
KAFKA_DIR     = f"/opt/kafka_{SCALA_VERSION}-{KAFKA_VERSION}"
KAFKA_URL     = f"https://archive.apache.org/dist/kafka/{KAFKA_VERSION}/kafka_{SCALA_VERSION}-{KAFKA_VERSION}.tgz"
KAFKA_ARCHIVE = "/tmp/kafka.tgz"
BIN           = f"{KAFKA_DIR}/bin"

def porta_aberta(porta):
    return subprocess.run(
        ["nc", "-z", "-w1", "localhost", str(porta)],
        capture_output=True
    ).returncode == 0

def aguardar_porta(porta, label, max_seg=35):
    print(f"   Aguardando {label}", end="")
    for _ in range(max_seg):
        time.sleep(1)
        print(".", end="", flush=True)
        if porta_aberta(porta):
            print(" OK")
            return True
    print(" ERROR")
    return False

# -- 1. Java --
print("-- 1/6  Java --")
if subprocess.run(["java", "-version"], capture_output=True).returncode != 0:
    print("   Instalando Java...")
    os.system("apt-get install -y -q default-jre-headless 2>/dev/null")
print("   Java OK")

# -- 2. Netcat --
print("-- 2/6  Netcat --")
if subprocess.run(["which", "nc"], capture_output=True).returncode != 0:
    print("   Instalando netcat...")
    os.system("apt-get install -y -q netcat-openbsd 2>/dev/null")
print("   Netcat OK")

# -- 3. kafka-python-ng --
print("-- 3/6  kafka-python-ng --")
try:
    import kafka
    print("   kafka-python-ng ja instalado")
except ModuleNotFoundError:
    print("   Instalando kafka-python-ng...")
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "kafka-python-ng", "--quiet"]
    )
    print("   kafka-python-ng instalado")

# -- 4. Kafka download + extracao --
print("-- 4/6  Kafka --")
start_sh = f"{BIN}/zookeeper-server-start.sh"

if os.path.isfile(start_sh):
    print(f"   Kafka ja instalado em {KAFKA_DIR}")
else:
    print(f"   Baixando Kafka {KAFKA_VERSION} (~113 MB)...")
    os.system(f"rm -rf {KAFKA_DIR} {KAFKA_ARCHIVE}")

    ret = os.system(f"wget -q --show-progress {KAFKA_URL} -O {KAFKA_ARCHIVE}")

    if ret != 0 or not os.path.isfile(KAFKA_ARCHIVE):
        raise RuntimeError("Download falhou. Verifique a conexao.")

    size_mb = os.path.getsize(KAFKA_ARCHIVE) / (1024 * 1024)
    print(f"   Arquivo baixado: {size_mb:.1f} MB")
    if size_mb < 50:
        raise RuntimeError(f"Arquivo muito pequeno ({size_mb:.1f} MB) — download incompleto.")

    print("   Extraindo...")
    os.system(f"tar -xzf {KAFKA_ARCHIVE} -C /opt/")

    if not os.path.isfile(start_sh):
        raise RuntimeError(f"Extracao falhou — {start_sh} nao encontrado.")

    print(f"   Kafka extraido com sucesso em {KAFKA_DIR}")

# -- 5. Configuracao do broker --
print("-- 5/6  Configuracao do Broker --")
os.system("pkill -f kafka.Kafka 2>/dev/null")
os.system("pkill -f zookeeper 2>/dev/null")
time.sleep(3)
os.system("rm -rf /tmp/kafka-logs /tmp/zookeeper")

server_props = f"{KAFKA_DIR}/config/server.properties"
os.system(f"sed -i '/^advertised\\.listeners/d' {server_props}")
os.system(f"sed -i '/^listeners=/d' {server_props}")
os.system(f"sed -i '/^log\\.dirs/d' {server_props}")
with open(server_props, "a") as f:
    f.write("\nlisteners=PLAINTEXT://0.0.0.0:9092\n")
    f.write("advertised.listeners=PLAINTEXT://localhost:9092\n")
    f.write("log.dirs=/tmp/kafka-logs\n")

check = subprocess.run(
    ["grep", "-E", "^listeners=|^advertised", server_props],
    capture_output=True, text=True
)
print(f"   {check.stdout.strip().replace(chr(10), ' | ')}")
print("   server.properties OK")

# -- 6. Servicos --
print("-- 6/6  Servicos --")

subprocess.Popen(
    f"{BIN}/zookeeper-server-start.sh {KAFKA_DIR}/config/zookeeper.properties".split(),
    stdout=open("/tmp/zookeeper.log", "w"),
    stderr=subprocess.STDOUT,
    preexec_fn=os.setsid
)
if not aguardar_porta(2181, "ZooKeeper"):
    os.system("tail -20 /tmp/zookeeper.log")
    raise RuntimeError("ZooKeeper nao subiu.")

subprocess.Popen(
    f"{BIN}/kafka-server-start.sh {KAFKA_DIR}/config/server.properties".split(),
    stdout=open("/tmp/kafka.log", "w"),
    stderr=subprocess.STDOUT,
    preexec_fn=os.setsid
)
if not aguardar_porta(9092, "Kafka Broker", max_seg=35):
    os.system("tail -20 /tmp/kafka.log")
    raise RuntimeError("Kafka Broker nao subiu.")

# -- Imports --
import json, threading, uuid, random
from datetime import datetime, timezone, timedelta
from collections import defaultdict, deque
from kafka import KafkaProducer, KafkaConsumer, KafkaAdminClient
from kafka.admin import NewTopic
from kafka.errors import TopicAlreadyExistsError

# -- Especifico Demo 4: topicos + helpers --
BOOTSTRAP = "localhost:9092"

TOPICOS = [
    NewTopic("eventos-confiabilidade", num_partitions=1, replication_factor=1),
    NewTopic("eventos-com-atraso",     num_partitions=1, replication_factor=1),
    NewTopic("dead-letter-queue",      num_partitions=1, replication_factor=1),
    NewTopic("eventos-idempotentes",   num_partitions=1, replication_factor=1),
]

admin = KafkaAdminClient(bootstrap_servers=BOOTSTRAP)
for t in TOPICOS:
    try:
        admin.create_topics([t])
        print(f"   Topico criado  : {t.name}")
    except TopicAlreadyExistsError:
        print(f"   Topico existente: {t.name}")

def make_producer(**kwargs):
    return KafkaProducer(
        bootstrap_servers=BOOTSTRAP,
        value_serializer=lambda v: json.dumps(v, default=str).encode('utf-8'),
        key_serializer=lambda k: k.encode('utf-8') if k else None,
        **kwargs
    )

def make_consumer(topic, group_id, reset='latest'):
    return KafkaConsumer(
        topic,
        bootstrap_servers=BOOTSTRAP,
        group_id=group_id,
        auto_offset_reset=reset,
        enable_auto_commit=True,
        value_deserializer=lambda b: json.loads(b.decode('utf-8')),
        key_deserializer=lambda b: b.decode('utf-8') if b else None,
        consumer_timeout_ms=4000,
    )

print("\n" + "=" * 42)
print("Kafka pronto! Pode continuar.")
print("=" * 42)
print("\nTopicos da Demo 4:")
print("   eventos-confiabilidade -> at-most/at-least/exactly-once")
print("   eventos-com-atraso     -> event time vs processing time")
print("   dead-letter-queue      -> isolamento de erros")
print("   eventos-idempotentes   -> deduplicacao por event_id")

---
## 🔒 Etapa 1 — Semânticas de Entrega

| Semântica | Descrição | Risco | Configuração |
|---|---|---|---|
| **At-most-once** | Entrega no máximo 1 vez (pode perder) | Perda de dados | `acks=0`, sem retry |
| **At-least-once** | Entrega ao menos 1 vez (pode duplicar) | Duplicatas | `acks=all`, com retry |
| **Exactly-once** | Entrega exatamente 1 vez | Nenhum | Transações Kafka |

In [ ]:
# ============================================================
# ETAPA 1.1 — AT-MOST-ONCE (fire and forget)
# ============================================================
# acks=0: o producer NÃO aguarda confirmação do broker.
# Máxima performance, mas sem garantia de entrega.
# Ideal para: logs de baixa criticidade, métricas de uso.

producer_amo = make_producer(acks=0, retries=0)

print(" AT-MOST-ONCE (acks=0, retries=0)")
print("   → Máxima performance, sem garantia de entrega\n")

enviados = 0
for i in range(10):
    evento = {"seq": i, "tipo": "at-most-once", "ts": datetime.now(timezone.utc).isoformat()}
    producer_amo.send("eventos-confiabilidade", key="amo", value=evento)
    enviados += 1

producer_amo.flush()
print(f"  {enviados} mensagens enviadas (sem confirmação aguardada)")
print(f"  Em caso de falha de rede, mensagens seriam PERDIDAS silenciosamente")

In [ ]:
# ============================================================
# ETAPA 1.2 — AT-LEAST-ONCE
# ============================================================
# acks='all' + retries: garante entrega, mas pode gerar duplicatas
# se o broker confirmar mas o producer não receber a confirmação.
# O CONSUMIDOR precisa ser idempotente para tratar duplicatas.

producer_alo = make_producer(acks='all', retries=5,
                              retry_backoff_ms=100)

print(" AT-LEAST-ONCE (acks='all', retries=5)")
print("   → Garantia de entrega, possível duplicata\n")

for i in range(10):
    evento = {
        "seq":           i,
        "tipo":          "at-least-once",
        "idempotency_key": f"alo-{i:04d}",  # Chave para deduplicação no consumidor
        "ts":            datetime.now(timezone.utc).isoformat()
    }
    future = producer_alo.send("eventos-confiabilidade", key="alo", value=evento)
    meta   = future.get(timeout=5)  # Aguarda confirmação explícita
    print(f"  Confirmado: seq={i} → partição {meta.partition}, offset {meta.offset}")

producer_alo.flush()
print(f"\n  'idempotency_key' permite que o consumidor detecte e descarte duplicatas")

In [ ]:
# ============================================================
# ETAPA 1.3 — EXACTLY-ONCE (Producer com Sequência Manual)
# ============================================================
# O kafka-python-ng não implementa enable_idempotence.
# Simulamos o comportamento: cada mensagem carrega um
# sequence_number único. O broker (ou consumer) usa esse
# número para detectar e descartar retransmissões duplicadas.
# É o mesmo mecanismo que o producer idempotente Java usa
# internamente — só que explícito aqui para fins didáticos.

producer_eo = make_producer(acks='all', retries=5,
                             max_in_flight_requests_per_connection=1)
# max_in_flight=1 garante ordenação estrita —
# sem isso retries podem inverter a ordem das mensagens

print("EXACTLY-ONCE simulado (acks='all' + sequence_number)")
print("   → Sequência única por mensagem permite dedup no broker/consumer\n")

seq_counter = 0  # Contador global de sequência do producer

for i in range(10):
    seq_counter += 1
    evento = {
        "seq":             seq_counter,         # Número de sequência incremental
        "producer_id":     "producer-demo4",    # ID fixo do producer
        "idempotency_key": f"demo4-{seq_counter:06d}",  # Chave composta para dedup
        "tipo":            "exactly-once",
        "ts":              datetime.now(timezone.utc).isoformat()
    }
    future = producer_eo.send("eventos-idempotentes", key="eo", value=evento)
    meta   = future.get(timeout=5)
    print(f"  seq={seq_counter:2d} | idempotency_key={evento['idempotency_key']} "
          f"→ offset {meta.offset}")

producer_eo.flush()
print("\n  Consumer usa (producer_id + seq) para detectar duplicatas")
print("  Se o producer reenviar seq=3, o consumer ignora — já viu esse par")

---
## ⏱️ Etapa 2 — Event Time vs Processing Time

In [ ]:
# ============================================================
# ETAPA 2 — EVENT TIME vs PROCESSING TIME
# ============================================================
# Event Time    = quando o evento ACONTECEU (timestamp do dado)
# Processing Time = quando o evento FOI PROCESSADO (agora)
#
# Diferença: eventos podem chegar ATRASADOS por rede lenta,
# dispositivos offline, etc. Processar por processing_time
# pode colocar eventos na janela errada.

producer = make_producer(acks='all')

# Simular eventos: alguns com atraso de até 5 minutos
agora = datetime.now(timezone.utc)
eventos = []

for i in range(15):
    atraso_s = random.choice([0, 0, 0, 30, 60, 120, 300])  # 70% sem atraso
    event_time = agora - timedelta(seconds=atraso_s)
    eventos.append({
        "evento_id":      f"evt-{i:03d}",
        "valor":          round(random.uniform(10, 1000), 2),
        "event_time":     event_time.isoformat(),          # Quando aconteceu
        "processing_time": agora.isoformat(),              # Quando foi enviado
        "atraso_segundos": atraso_s,
    })

for e in eventos:
    producer.send("eventos-com-atraso", value=e)
producer.flush()

print(" 15 eventos enviados com atrasos variados:\n")
print(f"{'Evento':<10} {'Event Time':<30} {'Atraso':>10}")
print("-" * 52)
for e in sorted(eventos, key=lambda x: x['event_time']):
    status = "✅" if e['atraso_segundos'] == 0 else "⚠️ "
    print(f"{status} {e['evento_id']:<8} {e['event_time'][:26]:<28} "
          f"{e['atraso_segundos']:>6}s atraso")

In [ ]:
# ============================================================
# COMPARAÇÃO: PROCESSAR POR PROCESSING TIME vs EVENT TIME
# ============================================================

consumer_tempo = make_consumer("eventos-com-atraso", "tempo-comparacao", reset='earliest')
msgs = list(consumer_tempo)
consumer_tempo.close()

WINDOW_MIN = 1  # Janela de 1 minuto

by_proc_time  = defaultdict(list)
by_event_time = defaultdict(list)

for msg in msgs:
    e = msg.value
    # Janela por PROCESSING TIME
    pt  = datetime.fromisoformat(e['processing_time'])
    pk  = pt.replace(second=0, microsecond=0).isoformat()[:16]
    by_proc_time[pk].append(e['valor'])

    # Janela por EVENT TIME
    et  = datetime.fromisoformat(e['event_time'])
    ek  = et.replace(second=0, microsecond=0).isoformat()[:16]
    by_event_time[ek].append(e['valor'])

print(" COMPARAÇÃO DE JANELAS: PROCESSING TIME vs EVENT TIME\n")
print(f"{'Janela':<20} {'ProcessingTime $':>18} {'EventTime $':>15}")
print("-" * 56)

all_windows = sorted(set(list(by_proc_time.keys()) + list(by_event_time.keys())))
for w in all_windows:
    pt_total = sum(by_proc_time.get(w, []))
    et_total = sum(by_event_time.get(w, []))
    delta = abs(pt_total - et_total)
    flag = "   DIVERGÊNCIA" if delta > 0.01 else ""
    print(f"  {w:<18} {pt_total:>16,.2f} {et_total:>14,.2f}{flag}")

print("\n Janelas diferentes = relatórios diferentes = decisões diferentes!")
print("   Em produção, SEMPRE processe por Event Time quando possível.")

---
## 🌊 Etapa 3 — Eventos Tardios e Watermarks

In [ ]:
# ============================================================
# ETAPA 3 — WATERMARK
# ============================================================
# Watermark = estimativa do tempo máximo de atraso tolerado.
# Define: "eventos com atraso > watermark serão descartados ou
#          tratados como late events"
#
# Watermark(t) = max(event_time visto até agora) - tolerancia

class WatermarkProcessor:
    """Processa eventos com tolerância de atraso via watermark."""

    def __init__(self, window_sec=60, late_tolerance_sec=30):
        self.window_sec          = window_sec
        self.late_tolerance_sec  = late_tolerance_sec
        self.max_event_time_seen = 0.0
        self.windows             = defaultdict(list)
        self.late_count          = 0
        self.dropped_count       = 0
        self.processed_count     = 0

    @property
    def watermark(self):
        return self.max_event_time_seen - self.late_tolerance_sec

    def process(self, event):
        try:
            et = datetime.fromisoformat(event['event_time']).timestamp()
        except Exception:
            et = time.time()

        # Atualizar max event time
        self.max_event_time_seen = max(self.max_event_time_seen, et)

        # Determinar janela do evento
        win_start = (et // self.window_sec) * self.window_sec
        win_end   = win_start + self.window_sec

        # Evento está abaixo do watermark?
        if win_end < self.watermark:
            self.dropped_count += 1
            return "DROPPED", win_start

        # Evento chegou atrasado mas dentro da tolerância?
        is_late = et < (self.max_event_time_seen - 5)  # > 5s de atraso
        if is_late:
            self.late_count += 1

        self.windows[win_start].append(event['valor'])
        self.processed_count += 1
        return "LATE" if is_late else "ON_TIME", win_start

    def get_window_stats(self):
        stats = []
        for ws, valores in sorted(self.windows.items()):
            stats.append({
                "window": datetime.fromtimestamp(ws, tz=timezone.utc).strftime("%H:%M:%S"),
                "count":  len(valores),
                "total":  round(sum(valores), 2),
            })
        return stats

# Processar eventos do tópico com atrasos
processor = WatermarkProcessor(window_sec=60, late_tolerance_sec=30)
consumer_wm = make_consumer("eventos-com-atraso", "watermark-proc", reset='earliest')

print(f" Processando com Watermark (tolerância: {processor.late_tolerance_sec}s)\n")
print(f"{'Evento':<10} {'Status':<12} {'Janela':<12} {'Watermark':<22} {'Atraso':>8}")
print("-" * 66)

for msg in consumer_wm:
    e      = msg.value
    status, win_start = processor.process(e)
    wm_str = datetime.fromtimestamp(processor.watermark, tz=timezone.utc).strftime("%H:%M:%S") \
             if processor.watermark > 0 else "N/A"
    win_str = datetime.fromtimestamp(win_start, tz=timezone.utc).strftime("%H:%M:%S")
    icon    = "✅" if status == "ON_TIME" else ("⚠️ " if status == "LATE" else "🗑️ ")
    print(f"{icon} {e['evento_id']:<8} {status:<12} {win_str:<12} {wm_str:<20} "
          f"{e['atraso_segundos']:>5}s")

consumer_wm.close()

print(f"\n RESULTADO DO WATERMARK:")
print(f"   On-time   : {processor.processed_count - processor.late_count}")
print(f"   Late      : {processor.late_count}")
print(f"   Descartados: {processor.dropped_count}")
print(f"\n   Estatísticas por janela:")
for s in processor.get_window_stats():
    print(f"   [{s['window']}] {s['count']} eventos | R$ {s['total']:,.2f}")

---
## 📬 Etapa 4 — Dead Letter Queue (DLQ)

In [ ]:
# ============================================================
# ETAPA 4 — DEAD LETTER QUEUE
# ============================================================
# Padrão essencial em produção: mensagens que falham no
# processamento NÃO devem bloquear o stream principal.
# Elas vão para uma DLQ (Dead Letter Queue) para análise posterior.

import time as _time

producer_main = make_producer(acks='all')
producer_dlq  = make_producer(acks='all')

TOPICO_ENTRADA = "eventos-confiabilidade"
GROUP_DLQ      = f"dlq-consumer-{int(_time.time())}"  # group_id unico por execucao

eventos_mistos = []
for i in range(20):
    if i % 5 == 0:
        eventos_mistos.append({"id": f"evt-{i}", "valor": "INVALIDO", "ts": "nao-e-data"})
    elif i % 7 == 0:
        eventos_mistos.append({"id": f"evt-{i}"})
    else:
        eventos_mistos.append({
            "id":    f"evt-{i}",
            "valor": round(random.uniform(1, 1000), 2),
            "ts":    datetime.now(timezone.utc).isoformat()
        })

for e in eventos_mistos:
    producer_main.send(TOPICO_ENTRADA, key="dlq-test", value=e)
producer_main.flush()
invalidos = sum(1 for e in eventos_mistos if 'valor' not in e or e.get('valor') == 'INVALIDO')
print(f"{len(eventos_mistos)} eventos enviados ({invalidos} invalidos)")

consumer_dlq = make_consumer(TOPICO_ENTRADA, GROUP_DLQ, reset='earliest')

def processar_evento(evento):
    if 'valor' not in evento:
        raise ValueError(f"Campo 'valor' ausente: {evento}")
    if not isinstance(evento.get('valor'), (int, float)):
        raise TypeError(f"Campo 'valor' nao e numerico: {evento.get('valor')}")
    return evento['valor'] * 1.1

print("\nProcessando com DLQ...\n")
ok_count = dlq_count = 0

for msg in consumer_dlq:
    # Processar apenas mensagens desta etapa
    if msg.key != "dlq-test":
        continue

    evento = msg.value
    try:
        resultado = processar_evento(evento)
        ok_count += 1
        print(f"  OK  {evento['id']:<8} processado -> resultado: {resultado:.2f}")
    except (ValueError, TypeError, KeyError) as e:
        dlq_msg = {
            "payload_original": evento,
            "erro":             str(e),
            "erro_tipo":        type(e).__name__,
            "topico_origem":    msg.topic,
            "particao":         msg.partition,
            "offset":           msg.offset,
            "falhou_em":        datetime.now(timezone.utc).isoformat(),
        }
        producer_dlq.send("dead-letter-queue", key="dlq", value=dlq_msg)
        dlq_count += 1
        print(f"  DLQ {evento.get('id','?'):<8} -> DLQ | {type(e).__name__}: {e}")

producer_dlq.flush()
consumer_dlq.close()

print(f"\nRESULTADO:")
print(f"   Processados : {ok_count}")
print(f"   Para a DLQ  : {dlq_count}")
print(f"   Stream principal NAO foi bloqueado pelos eventos invalidos!")

---
## 🔁 Etapa 5 — Idempotência no Consumidor

In [ ]:
# ============================================================
# ETAPA 5 — DEDUPLICAÇÃO NO CONSUMIDOR (corrigida)
# ============================================================

producer_idem = make_producer(acks='all')

# Usar group_id único para ler do início sem conflito com etapas anteriores
GROUP_IDEM = f"idem-consumer-{int(time.time())}"

# Produzir eventos com campo 'valor' e event_id bem definido
print("Enviando 15 mensagens (5 únicos × 3 cópias cada)...")
eventos_unicos = []
for i in range(5):
    evento = {
        "event_id":    f"EVT-{i:04d}",
        "producer_id": "etapa5-dedup",      # identificador desta etapa
        "valor":       round(random.uniform(100, 1000), 2),
        "ts":          datetime.now(timezone.utc).isoformat(),
    }
    eventos_unicos.append(evento)
    for _ in range(3):                      # 3 cópias simulando retry
        producer_idem.send("eventos-idempotentes", key="idem", value=evento)

producer_idem.flush()
print(f"   ✅ {len(eventos_unicos) * 3} mensagens enviadas\n")

# Consumer idempotente — group_id único garante leitura do início
# sem herdar offset de etapas anteriores
consumer_idem = KafkaConsumer(
    "eventos-idempotentes",
    bootstrap_servers=BOOTSTRAP,
    group_id=GROUP_IDEM,
    auto_offset_reset="earliest",
    enable_auto_commit=True,
    value_deserializer=lambda b: json.loads(b.decode('utf-8')),
    key_deserializer=lambda b: b.decode('utf-8') if b else None,
    consumer_timeout_ms=5000,
)

seen_ids   = set()
processados = 0
duplicatas  = 0

print("Consumindo com deduplicação:\n")

for msg in consumer_idem:
    evento   = msg.value

    # Ignorar mensagens de etapas anteriores (sem os campos esperados)
    if evento.get("producer_id") != "etapa5-dedup":
        continue

    event_id = evento.get("event_id")

    if event_id in seen_ids:
        duplicatas += 1
        print(f" DUPLICATA ignorada : {event_id} (offset {msg.offset})")
        continue

    seen_ids.add(event_id)
    processados += 1
    print(f" NOVO processado   : {event_id} | "
          f"R$ {evento['valor']:.2f} (offset {msg.offset})")

consumer_idem.close()

print(f"\n RESULTADO DA DEDUPLICAÇÃO:")
print(f"   Mensagens recebidas    : {processados + duplicatas}")
print(f"   Únicas processadas  : {processados}")
print(f"   Duplicatas ignoradas: {duplicatas}")
if processados + duplicatas > 0:
    print(f"   Taxa de duplicação     : "
          f"{duplicatas / (processados + duplicatas) * 100:.0f}%")
print(f"\n   Em produção: persista 'seen_ids' no Redis com TTL = retenção do tópico")